# 07 — First honest 2013 baseline

This notebook trains the first leakage-safe 2013 CNN baseline.

Training uses only the deterministic internal training split.
Model selection, early stopping, and threshold selection use
only the internal validation split. The official test split is
not loaded into a model dataset or evaluated.

Inputs are DBZ and VEL from both sweeps, producing a
`4 x 120 x 240` PyTorch tensor.


In [1]:
from google.colab import drive

drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pathlib import Path

PACKAGE_VERSION = "0.1.6"
VALIDATION_FRACTION = 0.20
VALIDATION_SEED = 20260913
TRAINING_SEED = 20260913

BATCH_SIZE = 64
NUM_WORKERS = 4
MAX_EPOCHS = 25
EARLY_STOPPING_PATIENCE = 5
LEARNING_RATE = 1e-3

BACKUP_ROOT = Path(
    "/content/drive/MyDrive/TorNet_Backup"
)
PACKAGE_PATH = (
    BACKUP_ROOT
    / "packages"
    / (
        "tornet_detection-"
        f"{PACKAGE_VERSION}-py3-none-any.whl"
    )
)
MANIFESTS_ROOT = BACKUP_ROOT / "manifests"
DRIVE_ARCHIVE_PATH = (
    BACKUP_ROOT / "tornet_2013.tar.gz"
)
EXPERIMENT_DIRECTORY = (
    BACKUP_ROOT
    / "experiments"
    / "2013_baseline_v1"
)
NORMALIZATION_PATH = (
    EXPERIMENT_DIRECTORY
    / "normalization.json"
)
CHECKPOINT_PATH = (
    EXPERIMENT_DIRECTORY
    / "best_model.pt"
)
METRICS_PATH = (
    EXPERIMENT_DIRECTORY
    / "validation_metrics.json"
)
HISTORY_PATH = (
    EXPERIMENT_DIRECTORY
    / "training_history.csv"
)

LOCAL_ARCHIVE_PATH = Path(
    "/content/tornet_2013.tar.gz"
)
EXTRACTION_ROOT = Path(
    "/content/tornet_2013_baseline"
)
LOCAL_CHECKPOINT_PATH = Path(
    "/content/best_model.pt"
)

for required_path in (
    PACKAGE_PATH,
    MANIFESTS_ROOT,
    DRIVE_ARCHIVE_PATH,
    NORMALIZATION_PATH,
):
    if not required_path.exists():
        raise FileNotFoundError(
            f"Missing required path: "
            f"{required_path}"
        )

preexisting_outputs = [
    path
    for path in (
        CHECKPOINT_PATH,
        METRICS_PATH,
        HISTORY_PATH,
    )
    if path.exists()
]

if preexisting_outputs:
    raise FileExistsError(
        "Refusing to overwrite existing baseline "
        f"outputs: {preexisting_outputs}"
    )

print("package:", PACKAGE_PATH)
print("normalization:", NORMALIZATION_PATH)
print("experiment:", EXPERIMENT_DIRECTORY)


package: /content/drive/MyDrive/TorNet_Backup/packages/tornet_detection-0.1.6-py3-none-any.whl
normalization: /content/drive/MyDrive/TorNet_Backup/experiments/2013_baseline_v1/normalization.json
experiment: /content/drive/MyDrive/TorNet_Backup/experiments/2013_baseline_v1


In [3]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "netCDF4>=1.7",
        "scikit-learn>=1.5",
    ],
    check=True,
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-deps",
        "--force-reinstall",
        str(PACKAGE_PATH),
    ],
    check=True,
)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '--no-deps', '--force-reinstall', '/content/drive/MyDrive/TorNet_Backup/packages/tornet_detection-0.1.6-py3-none-any.whl'], returncode=0)

In [4]:
import json
import random

import numpy as np
import pandas as pd
import torch

import tornado_detection

if (
    tornado_detection.__version__
    != PACKAGE_VERSION
):
    raise RuntimeError(
        "Unexpected package version: "
        f"{tornado_detection.__version__}"
    )

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. Select a Colab GPU "
        "runtime, restart, and run all cells."
    )

random.seed(TRAINING_SEED)
np.random.seed(TRAINING_SEED)
torch.manual_seed(TRAINING_SEED)
torch.cuda.manual_seed_all(TRAINING_SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda")

normalization = json.loads(
    NORMALIZATION_PATH.read_text()
)

expected_normalization = {
    "package_version": PACKAGE_VERSION,
    "year": 2013,
    "official_source_split": "train",
    "model_split": "train",
    "validation_fraction": (
        VALIDATION_FRACTION
    ),
    "validation_seed": VALIDATION_SEED,
    "variables": ["DBZ", "VEL"],
    "channel_order": [
        "DBZ_sweep_0",
        "DBZ_sweep_1",
        "VEL_sweep_0",
        "VEL_sweep_1",
    ],
    "tensor_shape": [120, 240, 4],
    "training_frame_count": 11056,
    "training_file_count": 2764,
    "training_positive_frame_count": 445,
    "label_mismatch_count": 0,
}

mismatches = {
    key: {
        "expected": value,
        "actual": normalization.get(key),
    }
    for key, value
    in expected_normalization.items()
    if normalization.get(key) != value
}

if mismatches:
    raise AssertionError(
        "Normalization provenance mismatch: "
        f"{mismatches}"
    )

channel_means = np.asarray(
    normalization["means"],
    dtype=np.float32,
)
channel_stds = np.asarray(
    normalization[
        "standard_deviations"
    ],
    dtype=np.float32,
)

assert channel_means.shape == (4,)
assert channel_stds.shape == (4,)
assert np.isfinite(channel_means).all()
assert np.isfinite(channel_stds).all()
assert np.all(channel_stds > 0)

print(
    "tornado_detection:",
    tornado_detection.__version__,
)
print("torch:", torch.__version__)
print(
    "GPU:",
    torch.cuda.get_device_name(0),
)
print(
    "means:",
    channel_means.tolist(),
)
print(
    "stds:",
    channel_stds.tolist(),
)


tornado_detection: 0.1.6
torch: 2.11.0+cu128
GPU: NVIDIA A100-SXM4-40GB
means: [23.446088790893555, 23.32160186767578, -2.4620916843414307, -2.225398540496826]
stds: [14.11445426940918, 14.181056022644043, 16.997556686401367, 17.79329490661621]


In [5]:
from tornado_detection.data import (
    assign_model_splits,
    load_canonical_frame_index,
)

canonical_index = (
    load_canonical_frame_index(
        MANIFESTS_ROOT
    )
)
assigned_index = assign_model_splits(
    canonical_index,
    validation_fraction=(
        VALIDATION_FRACTION
    ),
    seed=VALIDATION_SEED,
)

year_index = assigned_index.loc[
    assigned_index["year"].eq(2013)
].copy()

train_index = (
    year_index.loc[
        year_index["model_split"].eq(
            "train"
        )
    ]
    .sort_values("frame_id")
    .reset_index(drop=True)
)
validation_index = (
    year_index.loc[
        year_index["model_split"].eq(
            "validation"
        )
    ]
    .sort_values("frame_id")
    .reset_index(drop=True)
)
official_test_index = (
    year_index.loc[
        year_index["model_split"].eq(
            "test"
        )
    ]
    .sort_values("frame_id")
    .reset_index(drop=True)
)

assert len(train_index) == 11_056
assert len(validation_index) == 2_936
assert len(official_test_index) == 2_292
assert int(
    train_index["frame_label"].sum()
) == 445
assert int(
    validation_index[
        "frame_label"
    ].sum()
) == 143
assert int(
    official_test_index[
        "frame_label"
    ].sum()
) == 157

internal_groups = pd.concat(
    [
        train_index[
            [
                "validation_group_key",
                "model_split",
            ]
        ],
        validation_index[
            [
                "validation_group_key",
                "model_split",
            ]
        ],
    ],
    ignore_index=True,
).drop_duplicates()

crossing_groups = (
    internal_groups.groupby(
        "validation_group_key"
    )["model_split"]
    .nunique()
)

assert int(
    (crossing_groups > 1).sum()
) == 0

print(
    "train frames:",
    f"{len(train_index):,}",
)
print(
    "train positives:",
    int(
        train_index[
            "frame_label"
        ].sum()
    ),
)
print(
    "validation frames:",
    f"{len(validation_index):,}",
)
print(
    "validation positives:",
    int(
        validation_index[
            "frame_label"
        ].sum()
    ),
)
print(
    "official test held out:",
    f"{len(official_test_index):,}",
)
print("crossing groups: 0")


train frames: 11,056
train positives: 445
validation frames: 2,936
validation positives: 143
official test held out: 2,292
crossing groups: 0


In [6]:
import shutil
import tarfile
import time

if LOCAL_ARCHIVE_PATH.exists():
    LOCAL_ARCHIVE_PATH.unlink()

if EXTRACTION_ROOT.exists():
    shutil.rmtree(EXTRACTION_ROOT)

if LOCAL_CHECKPOINT_PATH.exists():
    LOCAL_CHECKPOINT_PATH.unlink()

EXTRACTION_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)

copy_started = time.perf_counter()

shutil.copyfile(
    DRIVE_ARCHIVE_PATH,
    LOCAL_ARCHIVE_PATH,
)

copy_seconds = (
    time.perf_counter()
    - copy_started
)

required_members = set(
    pd.concat(
        [
            train_index[
                "archive_member"
            ],
            validation_index[
                "archive_member"
            ],
        ],
        ignore_index=True,
    ).unique()
)

extracted_members = set()
extraction_started = (
    time.perf_counter()
)

with tarfile.open(
    LOCAL_ARCHIVE_PATH,
    mode="r:gz",
) as archive:
    for member in archive:
        if (
            not member.isfile()
            or member.name
            not in required_members
        ):
            continue

        destination = (
            EXTRACTION_ROOT
            / member.name
        )
        destination.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        source_file = archive.extractfile(
            member
        )

        if source_file is None:
            raise RuntimeError(
                f"Could not extract "
                f"{member.name}"
            )

        with (
            source_file,
            destination.open("wb")
            as output_file,
        ):
            shutil.copyfileobj(
                source_file,
                output_file,
                length=1024 * 1024,
            )

        extracted_members.add(
            member.name
        )

extraction_seconds = (
    time.perf_counter()
    - extraction_started
)

missing_members = (
    required_members
    - extracted_members
)

if missing_members:
    raise RuntimeError(
        "Missing internal train/validation "
        f"members: "
        f"{sorted(missing_members)[:10]}"
    )

print(
    "copy seconds:",
    round(copy_seconds, 3),
)
print(
    "extracted files:",
    f"{len(extracted_members):,}",
)
print(
    "extraction seconds:",
    round(extraction_seconds, 3),
)


copy seconds: 8.984
extracted files: 3,498
extraction seconds: 10.788


In [7]:
import xarray as xr
from torch.utils.data import (
    DataLoader,
    Dataset,
)

from tornado_detection.data import (
    build_frame_tensor,
)


class RadarFrameDataset(Dataset):
    def __init__(
        self,
        rows: pd.DataFrame,
        root: Path,
        means: np.ndarray,
        stds: np.ndarray,
    ) -> None:
        self.rows = (
            rows[
                [
                    "frame_id",
                    "archive_member",
                    "frame_index",
                    "frame_label",
                ]
            ]
            .reset_index(drop=True)
            .copy()
        )
        self.root = root
        self.means = means.reshape(
            1,
            1,
            4,
        )
        self.stds = stds.reshape(
            1,
            1,
            4,
        )

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(
        self,
        index: int,
    ):
        row = self.rows.iloc[index]
        path = (
            self.root
            / str(row["archive_member"])
        )

        with xr.open_dataset(
            path,
            engine="netcdf4",
        ) as dataset:
            result = build_frame_tensor(
                dataset,
                int(row["frame_index"]),
            )

        expected_label = int(
            row["frame_label"]
        )

        if result.label != expected_label:
            raise AssertionError(
                "Manifest/NetCDF label mismatch "
                f"for {row['frame_id']}"
            )

        values = (
            result.values
            - self.means
        ) / self.stds

        values = np.nan_to_num(
            values,
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        ).astype(
            np.float32,
            copy=False,
        )

        tensor = torch.from_numpy(
            values
        ).permute(
            2,
            0,
            1,
        ).contiguous()

        label = torch.tensor(
            [expected_label],
            dtype=torch.float32,
        )

        return tensor, label


train_dataset = RadarFrameDataset(
    train_index,
    EXTRACTION_ROOT,
    channel_means,
    channel_stds,
)
validation_dataset = RadarFrameDataset(
    validation_index,
    EXTRACTION_ROOT,
    channel_means,
    channel_stds,
)

generator = torch.Generator()
generator.manual_seed(TRAINING_SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=generator,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
)

smoke_inputs, smoke_labels = next(
    iter(train_loader)
)

assert smoke_inputs.shape[1:] == (
    4,
    120,
    240,
)
assert torch.isfinite(
    smoke_inputs
).all()
assert set(
    smoke_labels.numpy()
    .astype(int)
    .reshape(-1)
    .tolist()
).issubset({0, 1})

print(
    "training batches:",
    len(train_loader),
)
print(
    "validation batches:",
    len(validation_loader),
)
print(
    "smoke batch:",
    tuple(smoke_inputs.shape),
)
print(
    "smoke finite:",
    bool(
        torch.isfinite(
            smoke_inputs
        ).all()
    ),
)


training batches: 173
validation batches: 46
smoke batch: (64, 4, 120, 240)
smoke finite: True


In [8]:
import torch.nn as nn


class RadarBaselineCNN(nn.Module):
    def __init__(self) -> None:
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(
                4,
                16,
                kernel_size=3,
                padding=1,
            ),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(
                16,
                32,
                kernel_size=3,
                padding=1,
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                padding=1,
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(
                64,
                128,
                kernel_size=3,
                padding=1,
            ),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
        )

    def forward(
        self,
        inputs: torch.Tensor,
    ) -> torch.Tensor:
        return self.classifier(
            self.features(inputs)
        )


model = RadarBaselineCNN().to(device)

train_positive_count = int(
    train_index["frame_label"].sum()
)
train_negative_count = (
    len(train_index)
    - train_positive_count
)
positive_weight = (
    train_negative_count
    / train_positive_count
)

loss_function = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        [positive_weight],
        dtype=torch.float32,
        device=device,
    )
)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
)

parameter_count = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print(
    "parameters:",
    f"{parameter_count:,}",
)
print(
    "positive weight:",
    round(positive_weight, 6),
)


parameters: 106,385
positive weight: 23.844944


In [9]:
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


def evaluate(loader):
    model.eval()

    losses = []
    labels = []
    probabilities = []

    with torch.no_grad():
        for inputs, targets in loader:
            inputs = inputs.to(
                device,
                non_blocking=True,
            )
            targets = targets.to(
                device,
                non_blocking=True,
            )

            logits = model(inputs)
            loss = loss_function(
                logits,
                targets,
            )

            losses.append(
                float(loss.item())
            )
            labels.extend(
                targets.cpu()
                .numpy()
                .reshape(-1)
                .tolist()
            )
            probabilities.extend(
                logits.sigmoid()
                .cpu()
                .numpy()
                .reshape(-1)
                .tolist()
            )

    labels_array = np.asarray(
        labels,
        dtype=np.int64,
    )
    probabilities_array = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    return {
        "loss": float(
            np.mean(losses)
        ),
        "pr_auc": float(
            average_precision_score(
                labels_array,
                probabilities_array,
            )
        ),
        "roc_auc": float(
            roc_auc_score(
                labels_array,
                probabilities_array,
            )
        ),
        "labels": labels_array,
        "probabilities": (
            probabilities_array
        ),
    }


history = []
best_pr_auc = -np.inf
epochs_without_improvement = 0
training_started = time.perf_counter()

for epoch in range(
    1,
    MAX_EPOCHS + 1,
):
    model.train()
    training_losses = []

    epoch_started = time.perf_counter()

    for inputs, targets in train_loader:
        inputs = inputs.to(
            device,
            non_blocking=True,
        )
        targets = targets.to(
            device,
            non_blocking=True,
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = model(inputs)
        loss = loss_function(
            logits,
            targets,
        )

        if not torch.isfinite(loss):
            raise AssertionError(
                "Non-finite training loss"
            )

        loss.backward()

        gradients_finite = all(
            parameter.grad is None
            or torch.isfinite(
                parameter.grad
            ).all().item()
            for parameter
            in model.parameters()
        )

        if not gradients_finite:
            raise AssertionError(
                "Non-finite gradients"
            )

        optimizer.step()

        training_losses.append(
            float(loss.item())
        )

    validation_result = evaluate(
        validation_loader
    )
    epoch_seconds = (
        time.perf_counter()
        - epoch_started
    )

    row = {
        "epoch": epoch,
        "train_loss": float(
            np.mean(training_losses)
        ),
        "validation_loss": (
            validation_result["loss"]
        ),
        "validation_pr_auc": (
            validation_result["pr_auc"]
        ),
        "validation_roc_auc": (
            validation_result["roc_auc"]
        ),
        "epoch_seconds": epoch_seconds,
    }
    history.append(row)

    print(
        f"epoch={epoch:02d} "
        f"train_loss="
        f"{row['train_loss']:.6f} "
        f"val_loss="
        f"{row['validation_loss']:.6f} "
        f"val_pr_auc="
        f"{row['validation_pr_auc']:.6f} "
        f"val_roc_auc="
        f"{row['validation_roc_auc']:.6f} "
        f"seconds={epoch_seconds:.1f}"
    )

    if (
        validation_result["pr_auc"]
        > best_pr_auc
    ):
        best_pr_auc = (
            validation_result["pr_auc"]
        )
        epochs_without_improvement = 0

        torch.save(
            {
                "model_state_dict": (
                    model.state_dict()
                ),
                "epoch": epoch,
                "validation_pr_auc": (
                    best_pr_auc
                ),
                "package_version": (
                    PACKAGE_VERSION
                ),
                "validation_seed": (
                    VALIDATION_SEED
                ),
                "training_seed": (
                    TRAINING_SEED
                ),
                "channel_means": (
                    channel_means.tolist()
                ),
                "channel_stds": (
                    channel_stds.tolist()
                ),
            },
            LOCAL_CHECKPOINT_PATH,
        )
    else:
        epochs_without_improvement += 1

    if (
        epochs_without_improvement
        >= EARLY_STOPPING_PATIENCE
    ):
        print(
            "Early stopping after "
            f"{epoch} epochs"
        )
        break

training_seconds = (
    time.perf_counter()
    - training_started
)

if not LOCAL_CHECKPOINT_PATH.is_file():
    raise RuntimeError(
        "No best checkpoint was written"
    )

checkpoint = torch.load(
    LOCAL_CHECKPOINT_PATH,
    map_location=device,
    weights_only=False,
)
model.load_state_dict(
    checkpoint["model_state_dict"]
)

print()
print(
    "best epoch:",
    checkpoint["epoch"],
)
print(
    "best validation PR-AUC:",
    checkpoint[
        "validation_pr_auc"
    ],
)
print(
    "total training seconds:",
    round(training_seconds, 3),
)


epoch=01 train_loss=1.042926 val_loss=1.148190 val_pr_auc=0.135249 val_roc_auc=0.794605 seconds=74.6
epoch=02 train_loss=0.938909 val_loss=1.092754 val_pr_auc=0.130061 val_roc_auc=0.811155 seconds=74.3
epoch=03 train_loss=0.902325 val_loss=1.194523 val_pr_auc=0.159739 val_roc_auc=0.817537 seconds=74.1
epoch=04 train_loss=0.891643 val_loss=1.179424 val_pr_auc=0.160723 val_roc_auc=0.816092 seconds=74.2
epoch=05 train_loss=0.843661 val_loss=1.089094 val_pr_auc=0.172916 val_roc_auc=0.828531 seconds=74.5
epoch=06 train_loss=0.821974 val_loss=1.038517 val_pr_auc=0.192910 val_roc_auc=0.842845 seconds=74.9
epoch=07 train_loss=0.796899 val_loss=1.086370 val_pr_auc=0.178269 val_roc_auc=0.839853 seconds=74.1
epoch=08 train_loss=0.776971 val_loss=1.052484 val_pr_auc=0.181268 val_roc_auc=0.848569 seconds=74.2
epoch=09 train_loss=0.759448 val_loss=0.988005 val_pr_auc=0.188311 val_roc_auc=0.855101 seconds=74.1
epoch=10 train_loss=0.749652 val_loss=1.010793 val_pr_auc=0.189429 val_roc_auc=0.852873 sec

In [10]:
from sklearn.metrics import (
    confusion_matrix,
    precision_recall_curve,
    precision_score,
    recall_score,
)

final_validation = evaluate(
    validation_loader
)

labels = final_validation["labels"]
probabilities = final_validation[
    "probabilities"
]

precision_curve, recall_curve, thresholds = (
    precision_recall_curve(
        labels,
        probabilities,
    )
)

threshold_f1 = (
    2
    * precision_curve[:-1]
    * recall_curve[:-1]
    / np.maximum(
        precision_curve[:-1]
        + recall_curve[:-1],
        1e-12,
    )
)

best_threshold_index = int(
    np.nanargmax(threshold_f1)
)
selected_threshold = float(
    thresholds[
        best_threshold_index
    ]
)

predictions = (
    probabilities
    >= selected_threshold
).astype(np.int64)

precision = float(
    precision_score(
        labels,
        predictions,
        zero_division=0,
    )
)
recall = float(
    recall_score(
        labels,
        predictions,
        zero_division=0,
    )
)
f1 = float(
    2
    * precision
    * recall
    / max(
        precision + recall,
        1e-12,
    )
)

matrix = confusion_matrix(
    labels,
    predictions,
    labels=[0, 1],
)
true_negative, false_positive, false_negative, true_positive = (
    int(value)
    for value in matrix.ravel()
)

print(
    "validation PR-AUC:",
    final_validation["pr_auc"],
)
print(
    "validation ROC-AUC:",
    final_validation["roc_auc"],
)
print(
    "selected threshold:",
    selected_threshold,
)
print("precision:", precision)
print("recall:", recall)
print("F1:", f1)
print(
    "confusion matrix:",
    matrix.tolist(),
)


validation PR-AUC: 0.27130175844976934
validation ROC-AUC: 0.9091247098765896
selected threshold: 0.8083812594413757
precision: 0.3104693140794224
recall: 0.6013986013986014
F1: 0.4095238095238095
confusion matrix: [[2602, 191], [57, 86]]


In [11]:
import datetime

history_frame = pd.DataFrame(history)

metrics = {
    "artifact_kind": (
        "2013_validation_metrics"
    ),
    "created_at_utc": (
        datetime.datetime.now(
            datetime.timezone.utc
        ).isoformat()
    ),
    "package_version": PACKAGE_VERSION,
    "year": 2013,
    "variables": ["DBZ", "VEL"],
    "channel_order": (
        normalization[
            "channel_order"
        ]
    ),
    "validation_fraction": (
        VALIDATION_FRACTION
    ),
    "validation_seed": (
        VALIDATION_SEED
    ),
    "training_seed": TRAINING_SEED,
    "train_frame_count": int(
        len(train_index)
    ),
    "train_positive_count": int(
        train_positive_count
    ),
    "validation_frame_count": int(
        len(validation_index)
    ),
    "validation_positive_count": int(
        validation_index[
            "frame_label"
        ].sum()
    ),
    "official_test_frame_count": int(
        len(official_test_index)
    ),
    "official_test_evaluated": False,
    "positive_weight": float(
        positive_weight
    ),
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "maximum_epochs": MAX_EPOCHS,
    "completed_epochs": int(
        len(history)
    ),
    "best_epoch": int(
        checkpoint["epoch"]
    ),
    "selected_threshold": (
        selected_threshold
    ),
    "validation_loss": float(
        final_validation["loss"]
    ),
    "validation_pr_auc": float(
        final_validation["pr_auc"]
    ),
    "validation_roc_auc": float(
        final_validation["roc_auc"]
    ),
    "validation_precision": precision,
    "validation_recall": recall,
    "validation_f1": f1,
    "true_negative": true_negative,
    "false_positive": false_positive,
    "false_negative": false_negative,
    "true_positive": true_positive,
    "parameter_count": int(
        parameter_count
    ),
    "training_seconds": float(
        training_seconds
    ),
}

EXPERIMENT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

shutil.copyfile(
    LOCAL_CHECKPOINT_PATH,
    CHECKPOINT_PATH,
)
HISTORY_PATH.write_text(
    history_frame.to_csv(
        index=False
    )
)
METRICS_PATH.write_text(
    json.dumps(
        metrics,
        indent=2,
        sort_keys=True,
    )
    + "\n"
)

print(
    json.dumps(
        metrics,
        indent=2,
        sort_keys=True,
    )
)
print()
print("wrote:", CHECKPOINT_PATH)
print("wrote:", HISTORY_PATH)
print("wrote:", METRICS_PATH)


{
  "artifact_kind": "2013_validation_metrics",
  "batch_size": 64,
  "best_epoch": 24,
  "channel_order": [
    "DBZ_sweep_0",
    "DBZ_sweep_1",
    "VEL_sweep_0",
    "VEL_sweep_1"
  ],
  "completed_epochs": 25,
  "created_at_utc": "2026-09-13T19:53:56.240803+00:00",
  "false_negative": 57,
  "false_positive": 191,
  "learning_rate": 0.001,
  "maximum_epochs": 25,
  "official_test_evaluated": false,
  "official_test_frame_count": 2292,
  "package_version": "0.1.6",
  "parameter_count": 106385,
  "positive_weight": 23.84494382022472,
  "selected_threshold": 0.8083812594413757,
  "train_frame_count": 11056,
  "train_positive_count": 445,
  "training_seconds": 1856.4755442980004,
  "training_seed": 20260913,
  "true_negative": 2602,
  "true_positive": 86,
  "validation_f1": 0.4095238095238095,
  "validation_fraction": 0.2,
  "validation_frame_count": 2936,
  "validation_loss": 0.7684074572247007,
  "validation_positive_count": 143,
  "validation_pr_auc": 0.27130175844976934,
  "validat

In [12]:
shutil.rmtree(EXTRACTION_ROOT)
LOCAL_ARCHIVE_PATH.unlink()
LOCAL_CHECKPOINT_PATH.unlink()

assert not EXTRACTION_ROOT.exists()
assert not LOCAL_ARCHIVE_PATH.exists()
assert not LOCAL_CHECKPOINT_PATH.exists()
assert CHECKPOINT_PATH.is_file()
assert HISTORY_PATH.is_file()
assert METRICS_PATH.is_file()

print(
    "Removed all Colab-local baseline artifacts"
)
print(
    "Preserved experiment outputs in:",
    EXPERIMENT_DIRECTORY,
)


Removed all Colab-local baseline artifacts
Preserved experiment outputs in: /content/drive/MyDrive/TorNet_Backup/experiments/2013_baseline_v1
